# SAC Collector — D4RL Tier Audit

**目的**：在 Plan A（`agent_step_*.pt` 训练过程切片做真 D4RL tier）开跑前盘点 Drive 上的 ckpt 文件，回答 3 个决策性问题：

| 问题 | 答案需要什么 |
|---|---|
| **Q1.** `cross_u10_regression/.../seed_46/` 真的存了 `agent_step_*.pt` 吗？多少个？ | `ls *.pt` + 文件大小 |
| **Q2.** 这些 step ckpt 的 SACConfig schema 与本地一致（`obs_dim=48`、`privileged_obs_dim=0`）？ | `torch.load` × 4 后比 config |
| **Q3.** 它们的 success rate 谱是否真是 `random → expert` 单调？（D4RL tier 划分的物理基础） | 30 ep deterministic eval × 4 step ckpt |

**预算**：~15 min L4（其中 30-ep eval × 4 ckpt ≈ 10 min CPU）。

**参考文档**：
- [`docs/arrival_v2_sac_collector_design.md`](../docs/arrival_v2_sac_collector_design.md) rev.2 §4.2 / §5.1
- 2026-05-24 session 决策：放弃路径 1（cross_u15 sensor-floored），改用同 seed 训练切片做真 D4RL tier

## 0. 环境检查

In [ ]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name:    {torch.cuda.get_device_name(0)}")

## 1. 挂载 Drive + cd 到项目根

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd && git log --oneline -1

## 2. Q1 — 自动发现所有 step ckpt

**期望**：`agent_step_<NNNNNNNN>.pt`（8 位 0 填充 env_step；step 不是整数 100k 因 `num_envs=6`）多个 + `agent_best.pt` / `agent_final.pt` / `agent_latest.pt`。

本 cell 用 regex 自动解析 step 号 → 存进 `step_ckpts: list[tuple[int, Path]]` 供下游 schema 校验 + 全 sweep eval 使用。

**失败模式**：
- 只有 `agent_best/final/latest.pt` 3 个文件 → `checkpoint_every_steps` 当时被设成 1M（或更大），**Plan A 不可行，需 fallback Plan B 或 Plan C**
- 部分 step ckpt 缺失 → Drive sync 中断或被清理过

In [ ]:
import re
from pathlib import Path

CKPT_DIR = Path("checkpoints/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46")
EXP_DIR  = Path("experiments/arrival_v2_prototype/cross_u10_regression/arrival_v2/sac_vanilla/s0_k4/seed_46")

print(f"CKPT_DIR exists: {CKPT_DIR.exists()}")
print(f"EXP_DIR  exists: {EXP_DIR.exists()}")
print()

step_ckpts: list[tuple[int, Path]] = []
if CKPT_DIR.exists():
    all_pts = sorted(CKPT_DIR.glob("*.pt"))
    print(f"Found {len(all_pts)} .pt files total.\n")

    step_pat = re.compile(r"^agent_step_(\d+)\.pt$")
    for p in all_pts:
        m = step_pat.match(p.name)
        if m:
            step_ckpts.append((int(m.group(1)), p))
    step_ckpts.sort(key=lambda x: x[0])

    print(f"Parsed {len(step_ckpts)} agent_step_*.pt ckpts.")
    if step_ckpts:
        print(f"  step range: {step_ckpts[0][0]:>8d}  →  {step_ckpts[-1][0]:>8d}")
        deltas = [step_ckpts[i+1][0] - step_ckpts[i][0] for i in range(len(step_ckpts)-1)]
        if deltas:
            print(f"  step cadence: min={min(deltas)}  median={sorted(deltas)[len(deltas)//2]}  max={max(deltas)}")
        print(f"\n  first 3:")
        for step, p in step_ckpts[:3]:
            print(f"    {p.name:32s}  {p.stat().st_size/1e6:.1f} MB")
        print(f"  ...")
        print(f"  last 3:")
        for step, p in step_ckpts[-3:]:
            print(f"    {p.name:32s}  {p.stat().st_size/1e6:.1f} MB")

    print(f"\nbest/final/latest:")
    for name in ("agent_best.pt", "agent_final.pt", "agent_latest.pt"):
        p = CKPT_DIR / name
        if p.exists():
            print(f"  ✅ {name:25s}  {p.stat().st_size/1e6:.1f} MB")
        else:
            print(f"  ❌ {name} missing")
else:
    print("⚠️ CKPT_DIR not found — Drive 上路径与 spec §2.5 不一致，检查 prototype 分支 sync 状态")

## 3. 读 trainer_state.json — 确认 `checkpoint_every_steps` 实际值

In [ ]:
import json

ts_path = EXP_DIR / "trainer_state.json"
if ts_path.exists():
    ts = json.loads(ts_path.read_text())
    tc = ts.get("train_config", {})
    print(f"total_env_steps:         {ts.get('env_step')}")
    print(f"checkpoint_every_steps:  {tc.get('checkpoint_every_steps')}")
    print(f"random_steps:            {tc.get('random_steps')}")
    print(f"update_after:            {tc.get('update_after')}")
    print(f"num_envs:                {tc.get('num_envs')}")
    print(f"history_length:          {ts.get('history_length')}")
    print(f"probe_layout:            {ts.get('probe_layout')}")
    print(f"reward_objective:        {ts.get('reward_objective')}")
    print(f"algorithm:               {ts.get('algorithm')}")
    print(f"best_agent_path:         {ts.get('best_agent_path')}")
    print(f"final_agent_path:        {ts.get('final_agent_path')}")
else:
    print(f"⚠️ {ts_path} not found")

## 4. Q2 — schema 一致性（全 ckpt）

加载所有 step ckpt 的 `payload["config"]`，验证：
1. `payload["config"]` 是合法 SACConfig dict
2. obs_dim / action_dim / privileged_obs_dim / hidden_dim / use_layernorm 跨**所有 step** 一致（schema 没漂移）

→ 结果存进 `step_configs: dict[int, (Path, dict)]` 供下游 eval sweep 复用。

In [ ]:
step_configs: dict[int, tuple[Path, dict]] = {}
for step, path in step_ckpts:
    payload = torch.load(str(path), map_location="cpu", weights_only=False)
    cfg = payload.get("config", {})
    step_configs[step] = (path, cfg)

if not step_configs:
    print("⚠️ no step ckpts — Plan A NO-GO from Q1 already")
else:
    fields_to_check = ("obs_dim", "action_dim", "privileged_obs_dim", "hidden_dim", "use_layernorm")
    summaries = {f: {cfg.get(f) for _, cfg in step_configs.values()} for f in fields_to_check}
    print(f"Loaded {len(step_configs)} ckpt configs.\n")
    all_consistent = True
    for f, vals in summaries.items():
        if len(vals) == 1:
            print(f"  ✅ {f:25s}  {next(iter(vals))}")
        else:
            print(f"  ⚠️ {f:25s}  {vals}  (DRIFT)")
            all_consistent = False
    if all_consistent:
        print(f"\n✅ schema 一致 across all {len(step_configs)} step ckpts.")
    else:
        print(f"\n⚠️ schema 漂移检测到 — Plan A 需细分（不能把不同 schema 的 ckpt 混进同一 D4RL benchmark）")

## 5. Q3 — Dense eval sweep：全 ckpt × N ep deterministic

对**所有** step ckpt 跑 deterministic eval（task 随机化，每 ep 重新采 init pose + flow phase），输出完整 training curve（success / oob / timeout vs env_step）。

**为什么不用 30 ep**：30 ep × 39 ckpt × ~50s ≈ 33 min L4 偏长。改用 **N=10 ep** 当默认（SE ≈ 16pp at p=0.5）— 对 tier 划分 (0% / 50% / 87% / 100%) 区分度足够，时间预算 **~12 min CPU**。如需精度高可改 `EVAL_EPISODES = 20` 重跑。

**输出**：
- `results: dict[int, dict]` — 每个 step ckpt 的 success/oob/timeout/mean_len
- inline plot — training curve（success / oob / timeout vs step）

In [ ]:
from types import SimpleNamespace
import numpy as np
import time
import matplotlib.pyplot as plt

from auv_nav.env import ObservationHistoryWrapper
from auv_nav.sac_policy import SACCheckpointPolicy
from scripts.train_utils import (
    make_env_config_overrides,
    make_planar_env,
    make_reset_options,
)

# 与 ckpt 训练 reset_options 严格一致（来自 trainer_state.json）
EVAL_FLOW = "wake_data/wake_v8_U1p00_Re150_D12p00_dx0p60_Ti5pct_1200f_roi.npy"
EVAL_EPISODES = 10   # bump to 20-30 for low-noise; 10 enough for tier picking
EVAL_ARGS = SimpleNamespace(
    probe_layout="s0",
    history_length=4,
    task_geometry="cross_stream",
    target_speed=1.5,
    difficulty=None,
    action_mode=None,
    speed_ratio=None,
    objective="arrival_v2",
    energy_cost_gain=None,
    safety_cost_gain=None,
)
ENV_OVERRIDES = make_env_config_overrides(EVAL_ARGS)
RESET_OPTIONS = make_reset_options(EVAL_ARGS)


def eval_ckpt(ckpt_path: Path, n_eps: int = EVAL_EPISODES) -> dict:
    policy = SACCheckpointPolicy.from_checkpoint(ckpt_path, device="cpu", deterministic=True)
    env = make_planar_env(
        EVAL_FLOW,
        history_length=EVAL_ARGS.history_length,
        probe_layout=EVAL_ARGS.probe_layout,
        env_config_overrides=ENV_OVERRIDES,
    )
    try:
        base_env = env.env if isinstance(env, ObservationHistoryWrapper) else env
        successes, oob, timeout, lengths = 0, 0, 0, []
        for ep in range(n_eps):
            obs, info = env.reset(seed=ep, options=RESET_OPTIONS)
            done = False
            ep_len = 0
            while not done:
                action = policy.act(base_env, obs)
                obs, _r, terminated, truncated, info = env.step(action)
                done = terminated or truncated
                ep_len += 1
            reason = str(info.get("reason", ""))
            if info.get("success", False):
                successes += 1
            if reason == "out_of_bounds":
                oob += 1
            elif reason == "timeout":
                timeout += 1
            lengths.append(ep_len)
        return {
            "success": successes / n_eps,
            "oob": oob / n_eps,
            "timeout": timeout / n_eps,
            "mean_len": float(np.mean(lengths)),
        }
    finally:
        env.close()


results: dict[int, dict] = {}
sorted_steps = sorted(step_configs.keys())
t0 = time.time()
for i, step in enumerate(sorted_steps):
    path, _ = step_configs[step]
    t_step = time.time()
    r = eval_ckpt(path)
    results[step] = r
    elapsed = time.time() - t0
    eta = elapsed / (i + 1) * (len(sorted_steps) - i - 1)
    print(f"[{i+1:2d}/{len(sorted_steps):2d}] step {step:>8}: "
          f"success={r['success']:.0%}  oob={r['oob']:.0%}  "
          f"timeout={r['timeout']:.0%}  mean_len={r['mean_len']:.0f}  "
          f"({time.time()-t_step:.1f}s, ETA {eta:.0f}s)")
print(f"\nTotal eval time: {time.time()-t0:.1f}s")

# --- Training curve plot ---
if results:
    steps_x = sorted(results.keys())
    success_y = [results[s]["success"] for s in steps_x]
    oob_y     = [results[s]["oob"] for s in steps_x]
    timeout_y = [results[s]["timeout"] for s in steps_x]
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(steps_x, success_y, "o-", label="success",      linewidth=2, color="tab:green")
    ax.plot(steps_x, oob_y,     "x--", label="out_of_bounds", alpha=0.6,  color="tab:red")
    ax.plot(steps_x, timeout_y, "+--", label="timeout",       alpha=0.6,  color="tab:orange")
    ax.set_xlabel("env_step")
    ax.set_ylabel("rate")
    ax.set_title(f"cross_u10 / sac_vanilla / s0_k4 / seed_46 — deterministic eval ({EVAL_EPISODES} ep)")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)
    ax.legend(loc="center right")
    plt.tight_layout()
    plt.show()

## 6. Plan A 可行性判定 + 自动选 4 档 ckpt

**判定规则**：

| 判定 | 条件 | 行动 |
|---|---|---|
| ✅ **PLAN A GO** | 全 sweep 跑通 + 能从 results 里选到 4 个不同 step，分别 closest to {0%, 50%, 85%, 100%} 且实际 4 档 success 跨度 ≥ 0.6 + expert ≥ 0.85 | 直接进 4-tier dataset 收集 |
| ⚠️ **PLAN A PARTIAL** | expert/random 端 OK 但中间 tier 缺失（e.g. cliff learning，medium tier 离 50% 太远） | 用实际选出的 4 ckpt 进数据收集，paper 里 honest 标 "tier 划分受 cliff 学习曲线影响" |
| ❌ **PLAN A NO-GO** | results 不足 / max success < 0.85 / span < 0.5 | 回退 Plan B（replay_latest.pkl → npz）或 Plan C（pure expert tier） |

**Auto-pick 逻辑**：对 4 个目标 success（`random=0`, `medium=0.5`, `medium_expert=0.85`, `expert=1.0`）各取 |success - target| 最小的 ckpt（避免重复选同一 step），并报告**实际 vs 目标**偏差。

In [ ]:
TIER_TARGETS = {
    "random":         0.00,
    "medium":         0.50,
    "medium_expert":  0.85,
    "expert":         1.00,
}

picked: dict[str, dict] = {}
used_steps: set[int] = set()
if not results:
    verdict = "❌ NO-GO — no eval results"
else:
    # Auto-pick: for each tier, take the closest unused ckpt to target success
    for tier_name, target in TIER_TARGETS.items():
        candidates = [
            (step, results[step]["success"], abs(results[step]["success"] - target))
            for step in results
            if step not in used_steps
        ]
        if not candidates:
            break
        step, success, gap = min(candidates, key=lambda x: x[2])
        path = step_configs[step][0]
        picked[tier_name] = {
            "step": step,
            "success": success,
            "target": target,
            "gap": gap,
            "path": path,
            "oob": results[step]["oob"],
            "timeout": results[step]["timeout"],
        }
        used_steps.add(step)

    # Print picks
    print("=== Auto-picked D4RL tier ckpts ===\n")
    print(f"{'tier':>15s}  {'step':>8s}  {'success':>8s}  {'target':>7s}  {'|Δ|':>6s}  {'oob':>5s}  {'timeout':>7s}  path")
    for tier_name, info in picked.items():
        print(f"  {tier_name:>13s}  {info['step']:>8d}  "
              f"{info['success']:>7.0%}  {info['target']:>6.0%}  "
              f"{info['gap']:>5.0%}  {info['oob']:>4.0%}  {info['timeout']:>6.0%}  "
              f"{info['path'].name}")

    # Verdict
    successes = [info["success"] for info in picked.values()]
    span = max(successes) - min(successes) if successes else 0.0
    expert_ok = max(successes) >= 0.85 if successes else False
    random_ok = min(successes) <= 0.15 if successes else False
    medium_gap = picked.get("medium", {}).get("gap", 1.0)
    medium_close = medium_gap <= 0.15  # tolerate ±15pp from 50%

    if expert_ok and random_ok and medium_close and span >= 0.6:
        verdict = f"✅ GO — 4 tiers cleanly separated (span={span:.0%}, medium |Δ|={medium_gap:.0%})"
    elif expert_ok and random_ok and span >= 0.6:
        verdict = (f"⚠️ PARTIAL — span {span:.0%} OK + endpoints OK but medium tier off-target "
                   f"(|Δ|={medium_gap:.0%}); usable, paper flag as cliff-learning artifact")
    else:
        verdict = (f"❌ NO-GO — span={span:.0%} expert_ok={expert_ok} random_ok={random_ok}; "
                   "fallback to Plan B/C")

print(f"\n{verdict}")

## 7. Fallback 路径准备：盘点 `replay_latest.pkl`（Plan B 备份）

如果 Plan A NO-GO，需要 fallback Plan B（`replay_latest.pkl` → npz 转 medium-replay）。这一节只做盘点（不转），让用户看到 pickle 是否就在 Drive 上。

In [ ]:
replay_path = EXP_DIR / "state" / "replay_latest.pkl"
if replay_path.exists():
    size_mb = replay_path.stat().st_size / 1e6
    print(f"✅ {replay_path}")
    print(f"   size: {size_mb:.1f} MB ({size_mb/1024:.2f} GB)")
    print(f"   → 若 Plan A NO-GO，可走 Plan B：写 scripts/replay_to_offline.py 转 npz")
else:
    print(f"⚠️ {replay_path} not found — Plan B 也不可用，只能走 Plan C（pure expert tier）")

## 8. 回传 main session

跑完后直接把以下三段输出**截图或贴文本**回 main session：

1. **cell 7 输出** — `Parsed N agent_step_*.pt ckpts.` 那一段（确认 ckpt 数 + cadence）
2. **cell 13 inline plot** — training curve 截图（可视化 cliff 位置 + tier 分布）
3. **cell 15 输出** — 自动选的 4 tier 表 + verdict 行（这是决定 Plan A 走不走的核心证据）
4. **cell 17 输出** — `replay_latest.pkl` size（Plan B 备份状态）

我据此决定是否进 4-tier dataset 收集 + 写收集脚本。